In [1]:
import pandas as pd
import numpy as np
import networkx as nx
#import matplotlib.pyplot as plt 

In [2]:
string_df = pd.read_csv("../data/raw/9606.protein.links.detailed.v12.0.txt", sep=" ")
string_df.head(1)

,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
0,9606.ENSP00000000233,9606.ENSP00000356607,0,0,0,45,134,0,81,173


In [3]:
string_alias = pd.read_csv("../data/raw/9606.protein.aliases.v12.0.txt", sep="\t")
string_alias.head(1)

,#string_protein_id,alias,source
0,9606.ENSP00000000233,2B6H,Ensembl_PDB


In [4]:
#Ensembl_HGNC_ensembl_gene_id
#Ensembl
#Ensembl_gene

string_alias[string_alias['source'] == 'Ensembl_gene'].shape

(19699, 3)

In [5]:
# Filter for data_quality 
string_df[(string_df['database'] > 400) & (string_df['experimental'] > 400)]

,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
423,9606.ENSP00000000233,9606.ENSP00000256682,0,0,0,128,477,500,60,756
546,9606.ENSP00000000233,9606.ENSP00000306010,0,0,0,49,780,500,57,888
945,9606.ENSP00000000233,9606.ENSP00000440005,0,0,0,107,711,500,61,862
1656,9606.ENSP00000000412,9606.ENSP00000221957,0,0,0,117,510,500,479,872
2131,9606.ENSP00000001008,9606.ENSP00000431512,0,0,0,62,523,500,146,783
...,...,...,...,...,...,...,...,...,...,...
13714642,9606.ENSP00000501277,9606.ENSP00000360811,0,0,0,0,648,500,995,999
13714662,9606.ENSP00000501277,9606.ENSP00000257818,0,0,0,0,980,500,999,999
13714784,9606.ENSP00000501277,9606.ENSP00000388940,0,0,0,55,802,500,662,964
13714879,9606.ENSP00000501277,9606.ENSP00000294339,0,0,0,0,922,800,995,999


In [6]:
filtered_string = string_df[string_df['combined_score'] > 700]
filtered_string.shape

(472000, 10)

In [7]:
filtered_string_ensembl_df = (
    filtered_string
    .merge(
        string_alias[string_alias['source'] == 'Ensembl_gene'].add_suffix('_prot1'), 
        left_on="protein1",
        right_on="#string_protein_id_prot1",
        validate='many_to_one'
    )
    .merge(
        string_alias[string_alias['source'] == 'Ensembl_gene'].add_suffix('_prot2'), 
        left_on="protein2",
        right_on="#string_protein_id_prot2",
        validate='many_to_one'
    )
    .merge(
        string_alias[string_alias['source'] == 'Ensembl_gene'].add_suffix('_sym1'), 
        left_on="protein1",
        right_on="#string_protein_id_sym1",
        validate='many_to_one',
        how='left'
    )
    .merge(
        string_alias[string_alias['source'] == 'Ensembl_gene'].add_suffix('_sym2'), 
        left_on="protein2",
        right_on="#string_protein_id_sym2",
        validate='many_to_one',
        how='left'
    )
    .drop(
        ['#string_protein_id_prot1','#string_protein_id_prot2','source_prot1','source_prot2','#string_protein_id_sym2','#string_protein_id_sym1'],
        axis='columns'  
    )
)

# number of rows does not change 
assert filtered_string_ensembl_df.shape[0] == filtered_string.shape[0] # FAILS as some have no symbol 
filtered_string_ensembl_df.head(2)

,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,alias_prot1,alias_prot2,alias_sym1,source_sym1,alias_sym2,source_sym2
0,9606.ENSP00000000233,9606.ENSP00000158762,0,0,0,47,91,0,814,825,ENSG00000004059,ENSG00000072818,ENSG00000004059,Ensembl_gene,ENSG00000072818,Ensembl_gene
1,9606.ENSP00000000233,9606.ENSP00000357048,0,0,0,79,271,500,260,718,ENSG00000004059,ENSG00000122218,ENSG00000004059,Ensembl_gene,ENSG00000122218,Ensembl_gene


In [8]:
columns_selection = [
    'protein1',
    'protein2',
    'combined_score',
    'alias_prot1',
    'alias_prot2'
]

filtered_string_ensembl_df[columns_selection].to_parquet("../data/quality_control/string_filtered.parquet",compression="zstd")